In [1]:
import os
import sys
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np
import coiled
from dask.distributed import Client, LocalCluster
import dask
import zarr
import gcsfs
import json
import yaml
import re
import logging
from IPython.core.magic import register_cell_magic
@register_cell_magic
def comment(line, cell):
    # This magic does nothing, effectively ignoring the cell's content
    pass
# Add app to path
sys.path.insert(0, "/home/stefan/CRS/CRS.ZarrPipelines/app")
from utils import ScoringConfig, get_thresholds, score
from domain import scoring

dask.config.set({"array.rechunk.method": "p2p"})
print(os.getcwd())

/home/stefan/CRS/CRS.ZarrPipelines/app/scripts


/home/stefan/CRS/CRS.ZarrPipelines/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/home/stefan/CRS/CRS.ZarrPipelines/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


## Data Paths

In [2]:
INPUT_PATH = 'gs://crs_climate_data_public/PHYSICAL_HAZARDS_WORKING_FOLDER/HS/zarrs/processed_unscored/HS.zarr/'
OUTPUT_PATH_1_to_5 = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_5/HS.zarr'
OUTPUT_PATH_1_to_3 = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_3/HS.zarr'
OUTPUT_PATH_1_to_10 = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_10/LS.zarr'
OUTPUT_PATH_1_to_100_norm = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_100_norm/HS.zarr'

# Spin up Coiled Cluster

In [3]:
%%comment
# Initialize cluster
cluster = coiled.Cluster(
    n_workers=15,
    scheduler_vm_types="e2-standard-8",
    worker_vm_types="e2-standard-4",
    region="europe-west8",
    name="wonder",
    shutdown_on_close=False,
    idle_timeout="2 hours"
)
client = Client(cluster)
cluster.adapt(minimum=15, maximum=20)
logging.info(f"Cluster setup complete: {client}")

## Hazard Data

In [4]:
ds = xr.open_zarr(INPUT_PATH)
ds

<xarray.Dataset> Size: 111MB
Dimensions:    (lat: 600, lon: 1439, metric: 1, scenario: 2, statistic: 4,
                time: 4)
Coordinates:
  * lat        (lat) float32 2kB -60.0 -59.75 -59.5 -59.25 ... 89.5 89.75 90.0
  * lon        (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.1 179.4 179.6
  * metric     (metric) <U4 16B 'p95'
  * scenario   (scenario) <U5 40B 'RCP45' 'RCP85'
  * statistic  (statistic) <U4 64B 'mean' 'max' 'min' 'std'
  * time       (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
Data variables:
    utci       (statistic, scenario, time, metric, lat, lon) float32 111MB dask.array<chunksize=(1, 1, 1, 1, 250, 250), meta=np.ndarray>

# GADM data

# Step 1 : Create Unscored data according to config instructions

In [5]:
# Initialize config
config = ScoringConfig()

Found config at: /home/stefan/CRS/CRS.ZarrPipelines/app/config/scoring.yaml


ScannerError: mapping values are not allowed here
  in "/home/stefan/CRS/CRS.ZarrPipelines/app/config/scoring.yaml", line 116, column 54

In [10]:
# Get all thresholds for LS
ls_all = config.get_all_thresholds('HS')
print("All LS thresholds:")
print(ls_all)

All LS thresholds:
{'hazard': 'Heat Stress', 'code': 'HS', 'metrics': {'p95': {'scoring.5_point': {'thresholds': [-inf, 9, 26, 32, 38, 46, inf], 'scores': [0, 1, 2, 3, 4, 5]}, 'scoring.10_point': {'thresholds': [-inf, 4.5, 9, 17.5, 26, 29, 32, 35, 38, 42, 46, 52, inf], 'scores': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}}}}


# 1-5

In [24]:
thresh_hs_5, scores_hs_5 = config.get_thresholds('HS', '5', threshold_type='thresholds')
thresh_hs_5

[-inf, 9, 26, 32, 38, 46, inf]

# 1-10

In [25]:
thresh_hs_10, scores_hs_10 = config.get_thresholds('HS', '10', threshold_type='thresholds')
thresh_hs_10

[-inf, 4.5, 9, 17.5, 26, 29, 32, 35, 38, 42, 46, 52, inf]

# Step 2: Score Data according to bins with digitize

## 1-5

### 2.1 Score HS

In [26]:
scored_hs_5 = scoring.score_zarr(
    ds['utci'],
    thresh_hs_5,
    scores_hs_5,
    metric = "p95"
)
scored_hs_5

Scoring array with shape (4, 2, 4, 1, 600, 1439)
Thresholds: [-inf, 9, 26, 32, 38, 46, inf]
Scores: [0, 1, 2, 3, 4, 5]
Setting metric coordinate to: p95


<xarray.Dataset> Size: 221MB
Dimensions:    (lat: 600, lon: 1439, scenario: 2, statistic: 4, time: 4)
Coordinates:
  * lat        (lat) float32 2kB -60.0 -59.75 -59.5 -59.25 ... 89.5 89.75 90.0
  * lon        (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.1 179.4 179.6
  * scenario   (scenario) <U5 40B 'RCP45' 'RCP85'
  * statistic  (statistic) <U4 64B 'mean' 'max' 'min' 'std'
  * time       (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
    metric     <U3 12B 'p95'
Data variables:
    score      (statistic, scenario, time, lat, lon) float64 221MB dask.array<chunksize=(1, 1, 1, 250, 250), meta=np.ndarray>

## 1-10

In [27]:
scored_hs_10 = scoring.score_zarr(
    ds['utci'],
    thresh_hs_10,
    scores_hs_10,
    metric = "p95"
)
scored_hs_10

Scoring array with shape (4, 2, 4, 1, 600, 1439)
Thresholds: [-inf, 4.5, 9, 17.5, 26, 29, 32, 35, 38, 42, 46, 52, inf]
Scores: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Setting metric coordinate to: p95


<xarray.Dataset> Size: 221MB
Dimensions:    (lat: 600, lon: 1439, scenario: 2, statistic: 4, time: 4)
Coordinates:
  * lat        (lat) float32 2kB -60.0 -59.75 -59.5 -59.25 ... 89.5 89.75 90.0
  * lon        (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.1 179.4 179.6
  * scenario   (scenario) <U5 40B 'RCP45' 'RCP85'
  * statistic  (statistic) <U4 64B 'mean' 'max' 'min' 'std'
  * time       (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
    metric     <U3 12B 'p95'
Data variables:
    score      (statistic, scenario, time, lat, lon) float64 221MB dask.array<chunksize=(1, 1, 1, 250, 250), meta=np.ndarray>

# Step 3: GADM Aggregations
## https://earthmover.io/blog/vector-datacube-pt1/